# تجربة التوليد — النسخة 2

**الفرق عن النسخة 1:** الأسنان في الرسم الأولي **أسنان حقيقية** من بيانات التدريب (ترخيص CC BY 4.0) بشكلها التشريحي وملمسها، ولونها من لون أسنان الشخص نفسه مع تبييض معتدل. نموذج التوليد يمرّ عليها مروراً خفيفاً لتوحيد الإضاءة فقط، ونقارن SD 1.5 مع **SDXL**. والفحص الآلي يقيس الآن **الواقعية** أيضاً: الملمس، واللون، والحواف.

**التشغيل:** Runtime ← Change runtime type ← **T4 GPU**، ثم شغّل الخلايا بالترتيب. المدة نحو 45–60 دقيقة (ينزّل نحو 12 GB من النماذج أول مرة).

**الصور:** نفس `photos.zip` من التجربة الأولى (صور من الإنترنت، لا صور مرضى).

In [ ]:
%cd /content
!nvidia-smi -L
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/alnsyryfwzy55-source/digital-smile-design-ai /content/repo
!pip -q install mediapipe diffusers accelerate onnxruntime
import os; assert os.path.exists('/content/repo/research/generation/run_v2.py'), 'clone failed: run this cell again'
print('ready')

In [ ]:
import os, zipfile, glob
from google.colab import files
PHOTO_DIR = '/content/photos'
if not glob.glob(PHOTO_DIR + '/**/*.*', recursive=True):
    up = files.upload()                       # اختر photos.zip
    for n in up:
        zipfile.ZipFile(n).extractall(PHOTO_DIR)
print(len(glob.glob(PHOTO_DIR + '/**/*.*', recursive=True)), 'files')

In [ ]:
%cd /content/repo/research/generation
# two separate runs: if SDXL fails, the SD 1.5 results are already saved
!mkdir -p /content/results_v2
!python -u run_v2.py --photos /content/photos --out /content/results_v2/sd15 --models sd15 2>&1 | tee /content/results_v2/sd15_log.txt | grep -e sweep -e case_ -e pass_rate -e Error -e Killed
!python -u run_v2.py --photos /content/photos --out /content/results_v2/sdxl --models sdxl 2>&1 | tee /content/results_v2/sdxl_log.txt | grep -e sweep -e case_ -e pass_rate -e Error -e Killed
!ls /content/results_v2/sd15 /content/results_v2/sdxl | head -5

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/results_v2/*/case_*_sheet.jpg'))[:6]:
    print(p.split('/')[-1], '— قبل | التصميم | أسنان حقيقية بلا توليد | بعد التوليد'); display(Image(p, width=1100))

In [ ]:
!cd /content && rm -f results_v2.zip && zip -qr results_v2.zip results_v2
from google.colab import files
files.download('/content/results_v2.zip')